In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

In [ ]:
# 1. Загрузка данных
train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')

# 2. Предобработка
def clean_text(text):
    if pd.isna(text):
        return ''
    # Замена всего кроме букв и цифр на пробелы
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text.lower())
    return text

train['FullDescription'] = train['FullDescription'].apply(clean_text)
test['FullDescription'] = test['FullDescription'].apply(clean_text)

# Замена пропусков в категориальных признаках на 'nan'
for col in ['LocationNormalized', 'ContractTime']:
    train[col] = train[col].fillna('nan')
    test[col] = test[col].fillna('nan')

# Извлечение TF-IDF признаков из текста
vectorizer = TfidfVectorizer(min_df=5, max_df=0.9)
X_train_text = vectorizer.fit_transform(train['FullDescription'])
X_test_text = vectorizer.transform(test['FullDescription'])

# One-hot кодирование категориальных признаков
enc = DictVectorizer()
X_train_categ = enc.fit_transform(train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test_categ = enc.transform(test[['LocationNormalized', 'ContractTime']].to_dict('records'))

# Объединение признаков
X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])

# Целевая переменная
y_train = train['SalaryNormalized']

# 3. Обучение гребневой регрессии
model = Ridge(alpha=1.0, random_state=241)
model.fit(X_train, y_train)

# 4. Прогнозы
y_pred = model.predict(X_test)
answer = f"{round(y_pred[0], 2)} {round(y_pred[1], 2)}"

print("Предсказания для первых двух тестовых объектов:", answer)

Предсказания для первых двух тестовых объектов: 56837.75 37993.63


In [3]:
with open("answer.txt", "w") as f:
    f.write(answer)